In [3]:
# ==============================================================================
# GREENXAI-PCOS V4 — BLOCK 1: INTEGRITY AUDIT, DIHEDRAL DEDUP & STRATIFIED SPLITS
# Paste into Kaggle Cell 1
# ==============================================================================

import subprocess, sys
for pkg in ["imagehash", "grad-cam", "shap", "captum", "torchinfo", "timm", "scipy", "openpyxl"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import os, glob, json, time, hashlib, shutil, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
import imagehash

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, matthews_corrcoef, balanced_accuracy_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OUT_DIR = "/kaggle/working/GreenXAI_V4_Outputs"
FIG_DIR = os.path.join(OUT_DIR, "figures_600dpi")
CKPT_DIR = os.path.join(OUT_DIR, "checkpoints")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

LOG_PATH = os.path.join(OUT_DIR, "block1_audit_log.txt")
log_lines = []
def log(msg=""):
    print(msg, flush=True)
    log_lines.append(str(msg))

log(f"Block 1 started | Device: {DEVICE} | PyTorch: {torch.__version__}")

def find_primary_dataset():
    candidates = [
        "/kaggle/input/datasets/shuvokarmaker/pcos-12k/PCOS",
        "/kaggle/input/pcos-xai-ultrasound-dataset/PCOS",
        "/kaggle/input/pcos-xai-ultrasound-dataset",
    ]
    for c in candidates:
        if os.path.isdir(os.path.join(c, "infected")) and os.path.isdir(os.path.join(c, "noninfected")):
            return c
    for hit in glob.glob("/kaggle/input/**/infected", recursive=True):
        parent = os.path.dirname(hit)
        if os.path.isdir(os.path.join(parent, "noninfected")):
            return parent
    raise FileNotFoundError("Primary PCOS dataset (infected/noninfected) not found in /kaggle/input!")

def load_external_pcosgen():
    ext_roots = [p for p in glob.glob("/kaggle/input/*") if "pcosgen" in os.path.basename(p).lower()]
    if not ext_roots:
        ext_roots = [p for p in glob.glob("/kaggle/input/datasets/*/*") if "pcosgen" in p.lower()]
    if not ext_roots:
        log("[WARN] PCOSGen external dataset not found in /kaggle/input!")
        return pd.DataFrame(columns=["path", "label", "sha256"])

    ext_root = ext_roots[0]
    log(f"Found External Dataset Root: {ext_root}")
    pos_exact = {"abnormal", "unhealthy", "pcos", "infected", "positive"}
    neg_exact = {"normal", "healthy", "noninfected", "non_pcos", "non-pcos", "negative"}
    img_exts = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

    records = []
    for root, _, files in os.walk(ext_root):
        folder_low = os.path.basename(root).strip().lower()
        if folder_low in pos_exact:
            lbl = 0
        elif folder_low in neg_exact:
            lbl = 1
        else:
            continue
        for f in sorted(files):
            if f.lower().endswith(img_exts):
                records.append({"path": os.path.join(root, f), "label": lbl})

    if not records:
        log("[WARN] No matching class folders found in External Dataset!")
        return pd.DataFrame(columns=["path", "label", "sha256"])

    df_ext = pd.DataFrame(records).drop_duplicates(subset=["path"]).reset_index(drop=True)
    shas = []
    for p in df_ext["path"]:
        with open(p, "rb") as f:
            shas.append(hashlib.sha256(f.read()).hexdigest())
    df_ext["sha256"] = shas
    df_ext = df_ext.drop_duplicates(subset=["sha256"]).reset_index(drop=True)
    log(f"External PCOSGen indexed: {len(df_ext)} unique images | Pos(0)={(df_ext['label']==0).sum()} | Healthy(1)={(df_ext['label']==1).sum()}")
    return df_ext

DATA_DIR = find_primary_dataset()
log(f"Primary Dataset Root: {DATA_DIR}")
df_ext_raw = load_external_pcosgen()

CLASS_MAP = {"infected": 0, "noninfected": 1}
raw_records = []
for folder_name, label in CLASS_MAP.items():
    for root, _, files in os.walk(os.path.join(DATA_DIR, folder_name)):
        for fname in sorted(files):
            if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                fpath = os.path.join(root, fname)
                with open(fpath, "rb") as f:
                    sha = hashlib.sha256(f.read()).hexdigest()
                raw_records.append({"path": fpath, "label": label, "sha256": sha})

df_raw = pd.DataFrame(raw_records)
sha_classes = df_raw.groupby("sha256")["label"].nunique()
conflict_shas = set(sha_classes[sha_classes > 1].index)
df_unique = df_raw[~df_raw["sha256"].isin(conflict_shas)].drop_duplicates(subset=["sha256"]).reset_index(drop=True)

log("\n==========================================================")
log("1. SHA-256 DEDUPLICATION & DEDUPLICATED RESOLUTION STATS")
log("==========================================================")
log(f"Raw total: {len(df_raw)} | Pos={(df_raw['label']==0).sum()} | Healthy={(df_raw['label']==1).sum()}")
log(f"Cross-class conflicts: {len(conflict_shas)} | Exact duplicates removed: {len(df_raw) - len(df_unique)}")
log(f"Unique retained: {len(df_unique)} | Pos={(df_unique['label']==0).sum()} | Healthy={(df_unique['label']==1).sum()}")

widths, heights, fsizes, aspect_ratios, mean_ints, std_ints, hist_feats = [], [], [], [], [], [], []
for p in df_unique["path"]:
    fsizes.append(os.path.getsize(p))
    im = Image.open(p).convert("L")
    w, h = im.size
    widths.append(w); heights.append(h); aspect_ratios.append(w / (h + 1e-8))
    arr = np.array(im, dtype=np.float32) / 255.0
    mean_ints.append(float(arr.mean())); std_ints.append(float(arr.std()))
    hist, _ = np.histogram(arr, bins=16, range=(0.0, 1.0), density=True)
    hist_feats.append(hist.tolist())

df_unique["width"] = widths
df_unique["height"] = heights
df_unique["aspect"] = aspect_ratios
df_unique["filesize_kb"] = np.array(fsizes) / 1024.0
df_unique["res_bucket"] = [f"{w}x{h}" for w, h in zip(widths, heights)]

for lbl, cname in [(0, "PCOM/PCOS-Positive"), (1, "Healthy")]:
    sub = df_unique[df_unique["label"] == lbl]
    log(f"  {cname}: W median={sub['width'].median():.0f} (mean={sub['width'].mean():.1f}, range={sub['width'].min()}-{sub['width'].max()}), "
        f"H median={sub['height'].median():.0f} (mean={sub['height'].mean():.1f}, range={sub['height'].min()}-{sub['height'].max()}), "
        f"Size median={sub['filesize_kb'].median():.1f} KB (range={sub['filesize_kb'].min():.1f}-{sub['filesize_kb'].max():.1f} KB)")

log("\n==========================================================")
log("2. 8-WAY DIHEDRAL pHASH CLUSTERING & THRESHOLD SWEEP")
log("==========================================================")
def compute_dihedral_phashes(pil_img):
    variants = [
        pil_img,
        pil_img.transpose(Image.FLIP_LEFT_RIGHT),
        pil_img.rotate(90, expand=True),
        pil_img.rotate(180, expand=True),
        pil_img.rotate(270, expand=True),
        pil_img.rotate(90, expand=True).transpose(Image.FLIP_LEFT_RIGHT),
        pil_img.rotate(180, expand=True).transpose(Image.FLIP_LEFT_RIGHT),
        pil_img.rotate(270, expand=True).transpose(Image.FLIP_LEFT_RIGHT),
    ]
    return np.array([imagehash.phash(v).hash.flatten() for v in variants], dtype=np.uint8)

N = len(df_unique)
base_hashes = np.zeros((N, 64), dtype=np.uint8)
all_dihedral = np.zeros((N, 8, 64), dtype=np.uint8)
for i, p in enumerate(df_unique["path"]):
    im = Image.open(p).convert("RGB")
    dh = compute_dihedral_phashes(im)
    base_hashes[i] = dh[0]
    all_dihedral[i] = dh

A = base_hashes.astype(np.float32)
B = all_dihedral.reshape(N * 8, 64).astype(np.float32)
A_sum = A.sum(axis=1, keepdims=True)
B_sum = B.sum(axis=1, keepdims=True).T

min_dist_mat = np.zeros((N, N), dtype=np.uint8)
CHUNK = 256
for i in range(0, N, CHUNK):
    end = min(i + CHUNK, N)
    dots = A[i:end] @ B.T
    dists = (A_sum[i:end] + B_sum - 2.0 * dots).reshape(end - i, N, 8)
    min_dist_mat[i:end] = np.clip(np.rint(dists.min(axis=2)), 0, 64).astype(np.uint8)
min_dist_mat = np.minimum(min_dist_mat, min_dist_mat.T)

def build_groups(dist_mat, thresh):
    parent = list(range(len(dist_mat)))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    rows, cols = np.where(np.triu(dist_mat <= thresh, k=1))
    for r, c in zip(rows, cols):
        ra, rb = find(int(r)), find(int(c))
        if ra != rb:
            parent[rb] = ra
    roots = [find(i) for i in range(len(dist_mat))]
    rmap = {g: idx for idx, g in enumerate(sorted(set(roots)))}
    return np.array([rmap[g] for g in roots], dtype=np.int32)

sweep_thresholds = [2, 4, 6, 8, 10, 12]
sweep_counts = []
for th in sweep_thresholds:
    g = build_groups(min_dist_mat, th)
    gsizes = pd.Series(g).value_counts()
    n_g = len(gsizes)
    sweep_counts.append((th, n_g, int(gsizes.max()), float(gsizes.mean()), int((gsizes == 1).sum())))
    log(f"  Dihedral pHash d_H <= {th:2d} | Groups: {n_g:4d} | Max size: {gsizes.max():3d} | Mean size: {gsizes.mean():.2f} | Singletons: {(gsizes==1).sum():4d}")

PRIMARY_THRESH = 6
df_unique["group_id"] = build_groups(min_dist_mat, PRIMARY_THRESH)
log(f"Selected Dihedral pHash threshold: d_H <= {PRIMARY_THRESH} -> {df_unique['group_id'].nunique()} groups")

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.plot([x[0] for x in sweep_counts], [x[1] for x in sweep_counts], marker="o", lw=2.2, color="#1F618D")
ax.axvline(PRIMARY_THRESH, color="#C0392B", linestyle="--", label=f"Selected ($d_H \\leq {PRIMARY_THRESH}$)")
ax.set_xlabel("Dihedral pHash Hamming Distance Threshold ($d_H$)")
ax.set_ylabel("Number of Distinct Groups")
ax.set_title("8-Way Dihedral pHash Deduplication Threshold Sweep")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "dedup_threshold_sweep.png"), dpi=600, bbox_inches="tight")
plt.close()

log("\n==========================================================")
log("3. STRATIFIED GROUP SPLIT (~71/14/14, BALANCED CLASS RATIO)")
log("==========================================================")
sgkf_split = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
split_labels = np.empty(N, dtype=object)
for f_idx, (_, te_f) in enumerate(sgkf_split.split(df_unique, df_unique["label"], df_unique["group_id"])):
    if f_idx == 0:
        split_labels[te_f] = "test"
    elif f_idx == 1:
        split_labels[te_f] = "val"
    else:
        split_labels[te_f] = "train"
df_unique["split"] = split_labels

sgkf_10 = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
cv_fold = np.zeros(N, dtype=int)
for f_idx, (_, te_f) in enumerate(sgkf_10.split(df_unique, df_unique["label"], df_unique["group_id"])):
    cv_fold[te_f] = f_idx
df_unique["cv_fold"] = cv_fold

tot_g = df_unique["group_id"].nunique()
for sp in ["train", "val", "test"]:
    sub = df_unique[df_unique["split"] == sp]
    ng = sub["group_id"].nunique()
    pos_n, neg_n = int((sub["label"] == 0).sum()), int((sub["label"] == 1).sum())
    log(f"  {sp.upper():<5} | Scans: {len(sub):4d} ({100*len(sub)/N:5.2f}%) [Pos={pos_n}, Healthy={neg_n} ({100*neg_n/len(sub):.1f}%)] | Groups: {ng:4d} ({100*ng/tot_g:5.2f}%)")

tr_g = set(df_unique[df_unique["split"] == "train"]["group_id"])
va_g = set(df_unique[df_unique["split"] == "val"]["group_id"])
te_g = set(df_unique[df_unique["split"] == "test"]["group_id"])
assert len(tr_g & va_g) == 0 and len(tr_g & te_g) == 0 and len(va_g & te_g) == 0, "Group leakage detected!"

log("\n==========================================================")
log("4. FROZEN EMBEDDING COPY-DETECTOR & k-NN PROBE (ResNet50)")
log("==========================================================")
class MaskUltrasoundBorders:
    def __call__(self, img):
        w, h = img.size
        img = img.crop((int(0.03 * w), int(0.03 * h), int(0.97 * w), int(0.97 * h)))
        arr = np.array(img, dtype=np.uint8)
        ch, cw = arr.shape[:2]
        arr[int(0.90 * ch):, int(0.82 * cw):] = 0
        arr[:int(0.06 * ch), :int(0.12 * cw)] = 0
        arr[:int(0.06 * ch), int(0.88 * cw):] = 0
        return Image.fromarray(arr)

probe_tf = transforms.Compose([
    MaskUltrasoundBorders(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class PathDS(Dataset):
    def __init__(self, paths, tf): self.paths, self.tf = list(paths), tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return self.tf(Image.open(self.paths[i]).convert("RGB"))

backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
backbone.fc = nn.Identity()
backbone = backbone.to(DEVICE).eval()

embs = []
with torch.no_grad():
    for xb in DataLoader(PathDS(df_unique["path"], probe_tf), batch_size=64, shuffle=False, num_workers=2):
        e = backbone(xb.to(DEVICE))
        e = torch.nn.functional.normalize(e, dim=1)
        embs.append(e.cpu().numpy())
embs = np.vstack(embs)

tr_mask = (df_unique["split"] == "train").values
te_mask = (df_unique["split"] == "test").values
sim_te_tr = embs[te_mask] @ embs[tr_mask].T
max_sims = sim_te_tr.max(axis=1)
log(f"  Closest Train-Test Cosine Similarity (Frozen ResNet50): "
    f"Mean={max_sims.mean():.4f}, Median={np.median(max_sims):.4f}, P95={np.percentile(max_sims, 95):.4f}, Max={max_sims.max():.4f}")
log(f"  Test scans with cosine sim > 0.95 to any train scan: {(max_sims > 0.95).sum()} / {len(max_sims)}")

knn = KNeighborsClassifier(n_neighbors=5, metric="cosine")
knn.fit(embs[tr_mask], df_unique.loc[tr_mask, "label"].values)
knn_pred = knn.predict(embs[te_mask])
knn_prob = knn.predict_proba(embs[te_mask])[:, 0]
y_te = df_unique.loc[te_mask, "label"].values
log(f"  Frozen ResNet50 5-NN Probe -> Acc: {accuracy_score(y_te, knn_pred)*100:.2f}% | "
    f"BalAcc: {balanced_accuracy_score(y_te, knn_pred)*100:.2f}% | MCC: {matthews_corrcoef(y_te, knn_pred):.4f} | "
    f"AUC: {roc_auc_score((y_te==0).astype(int), knn_prob):.4f}")

log("\n==========================================================")
log("5. METADATA-ONLY SHORTCUT BIAS AUDIT (STRATIFIED SPLIT)")
log("==========================================================")
X_meta = np.column_stack([widths, heights, aspect_ratios, fsizes, mean_ints, std_ints, np.array(hist_feats)])
y_all = df_unique["label"].values
for flbl, cols in [("File Metadata Only (W, H, Aspect, Size)", slice(0, 4)),
                   ("Metadata + Intensity Hist (22 feats)", slice(0, 22))]:
    for cname, clf in [
        ("LogisticRegression", make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED))),
        ("RandomForest", RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)),
    ]:
        clf.fit(X_meta[tr_mask, cols], y_all[tr_mask])
        pr = clf.predict(X_meta[te_mask, cols])
        pb = clf.predict_proba(X_meta[te_mask, cols])[:, 0]
        log(f"  {flbl:<40} | {cname:<18} | Acc: {accuracy_score(y_te, pr)*100:6.2f}% | MCC: {matthews_corrcoef(y_te, pr):.4f} | AUC: {roc_auc_score((y_te==0).astype(int), pb):.4f}")

log("\n==========================================================")
log("6. CROSS-DATASET OVERLAP AUDIT (PRIMARY vs EXTERNAL PCOSGEN)")
log("==========================================================")
if len(df_ext_raw) > 0:
    primary_shas = set(df_unique["sha256"])
    ext_exact_overlap = df_ext_raw["sha256"].isin(primary_shas)
    df_ext_clean = df_ext_raw[~ext_exact_overlap].reset_index(drop=True)

    ext_hashes = np.array([imagehash.phash(Image.open(p).convert("RGB")).hash.flatten() for p in df_ext_clean["path"]], dtype=np.uint8)
    E = ext_hashes.astype(np.float32)
    E_sum = E.sum(axis=1, keepdims=True)
    A_sum_T = A_sum.T
    phash_overlap = np.zeros(len(df_ext_clean), dtype=bool)
    for i in range(0, len(df_ext_clean), 256):
        end = min(i + 256, len(df_ext_clean))
        d = E_sum[i:end] + A_sum_T - 2.0 * (E[i:end] @ A.T)
        phash_overlap[i:end] = (d.min(axis=1) <= 6)
    df_ext_final = df_ext_clean[~phash_overlap].reset_index(drop=True)
    log(f"  External raw unique: {len(df_ext_raw)} | Exact SHA overlap removed: {ext_exact_overlap.sum()} | "
        f"pHash (d_H<=6) overlap removed: {phash_overlap.sum()} | Clean External retained: {len(df_ext_final)} "
        f"[Pos={(df_ext_final['label']==0).sum()}, Healthy={(df_ext_final['label']==1).sum()}]")
    df_ext_final.to_csv(os.path.join(OUT_DIR, "external_pcosgen_clean.csv"), index=False)
else:
    pd.DataFrame(columns=["path", "label", "sha256"]).to_csv(os.path.join(OUT_DIR, "external_pcosgen_clean.csv"), index=False)

manifest_path = os.path.join(OUT_DIR, "splits_v4_manifest.csv")
df_unique.to_csv(manifest_path, index=False)

mask_op = MaskUltrasoundBorders()
fig, axes = plt.subplots(4, 3, figsize=(9.5, 11.5))
samples_pos = df_unique[df_unique["label"] == 0]["path"].iloc[:2].tolist()
samples_neg = df_unique[df_unique["label"] == 1]["path"].iloc[:2].tolist()
for row_i, (pth, tag) in enumerate(zip(samples_pos + samples_neg, ["PCOM-Pos #1", "PCOM-Pos #2", "Healthy #1", "Healthy #2"])):
    raw_im = Image.open(pth).convert("RGB")
    masked_im = mask_op(raw_im)
    resized_im = masked_im.resize((224, 224), Image.BILINEAR)
    axes[row_i, 0].imshow(raw_im); axes[row_i, 0].set_title(f"{tag}: Raw ({raw_im.size[0]}x{raw_im.size[1]})", fontsize=9)
    axes[row_i, 1].imshow(masked_im); axes[row_i, 1].set_title("3% Crop + Corner Mask", fontsize=9)
    axes[row_i, 2].imshow(resized_im); axes[row_i, 2].set_title("Resized 224x224", fontsize=9)
    for c in range(3): axes[row_i, c].axis("off")
plt.suptitle("Preprocessing & Deterministic Border/Corner Masking (2 Examples per Class)", fontsize=11, y=0.995)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "preprocessing_pipeline.png"), dpi=600, bbox_inches="tight")
plt.close()

with open(LOG_PATH, "w") as f:
    f.write("\n".join(log_lines) + "\n")

log(f"\nBLOCK 1 COMPLETE! Saved manifest to {manifest_path}")


Block 1 started | Device: cuda | PyTorch: 2.11.0+cu130
Primary Dataset Root: /kaggle/input/datasets/shuvokarmaker/pcos-12k/PCOS
Found External Dataset Root: /kaggle/input/datasets/pcosgenaudit/pcosgen-deduplicated
External PCOSGen indexed: 225 unique images | Pos(0)=162 | Healthy(1)=63

1. SHA-256 DEDUPLICATION & DEDUPLICATED RESOLUTION STATS
Raw total: 11784 | Pos=6784 | Healthy=5000
Cross-class conflicts: 0 | Exact duplicates removed: 7788
Unique retained: 3996 | Pos=3184 | Healthy=812
  PCOM/PCOS-Positive: W median=275 (mean=271.3, range=112-343), H median=270 (mean=268.1, range=108-343), Size median=19.8 KB (range=5.0-43.6 KB)
  Healthy: W median=848 (mean=713.6, range=160-984), H median=578 (mean=620.0, range=160-984), Size median=57.7 KB (range=8.3-126.8 KB)

2. 8-WAY DIHEDRAL pHASH CLUSTERING & THRESHOLD SWEEP
  Dihedral pHash d_H <=  2 | Groups: 2383 | Max size: 133 | Mean size: 1.68 | Singletons: 1759
  Dihedral pHash d_H <=  4 | Groups: 1837 | Max size: 248 | Mean size: 2.18 

100%|██████████| 97.8M/97.8M [00:00<00:00, 232MB/s]


  Closest Train-Test Cosine Similarity (Frozen ResNet50): Mean=0.9081, Median=0.9174, P95=0.9535, Max=0.9666
  Test scans with cosine sim > 0.95 to any train scan: 17 / 162
  Frozen ResNet50 5-NN Probe -> Acc: 97.53% | BalAcc: 98.02% | MCC: 0.9494 | AUC: 0.9821

5. METADATA-ONLY SHORTCUT BIAS AUDIT (STRATIFIED SPLIT)
  File Metadata Only (W, H, Aspect, Size)  | LogisticRegression | Acc: 100.00% | MCC: 1.0000 | AUC: 1.0000
  File Metadata Only (W, H, Aspect, Size)  | RandomForest       | Acc: 100.00% | MCC: 1.0000 | AUC: 1.0000
  Metadata + Intensity Hist (22 feats)     | LogisticRegression | Acc: 100.00% | MCC: 1.0000 | AUC: 1.0000
  Metadata + Intensity Hist (22 feats)     | RandomForest       | Acc: 100.00% | MCC: 1.0000 | AUC: 1.0000

6. CROSS-DATASET OVERLAP AUDIT (PRIMARY vs EXTERNAL PCOSGEN)
  External raw unique: 225 | Exact SHA overlap removed: 91 | pHash (d_H<=6) overlap removed: 2 | Clean External retained: 132 [Pos=70, Healthy=62]

BLOCK 1 COMPLETE! Saved manifest to /kaggle

In [ ]:
# ==============================================================================
# GREENXAI-PCOS V4 — BLOCK 2: TRAINING, 5-SEED ABLATIONS, BASELINES, VAL GRID & 10-FOLD CV
# Paste into Kaggle Cell 2 | Runtime: ~3-4 hours on T4 GPU
# ==============================================================================

import os, json, time, copy, math, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import timm
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, balanced_accuracy_score, matthews_corrcoef
)
from scipy.stats import wilcoxon

t0_block2 = time.time()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_DIR = "/kaggle/working/GreenXAI_V4_Outputs"
CKPT_DIR = os.path.join(OUT_DIR, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

df = pd.read_csv(os.path.join(OUT_DIR, "splits_v4_manifest.csv"))
LOG_PATH = os.path.join(OUT_DIR, "block2_training_log.txt")
log_lines = []
def log(msg=""):
    print(msg, flush=True)
    log_lines.append(str(msg))

log(f"Block 2 started | Device: {DEVICE} | Total unique scans: {len(df)}")

class MaskUltrasoundBorders:
    def __call__(self, img):
        w, h = img.size
        img = img.crop((int(0.03 * w), int(0.03 * h), int(0.97 * w), int(0.97 * h)))
        arr = np.array(img, dtype=np.uint8)
        ch, cw = arr.shape[:2]
        arr[int(0.90 * ch):, int(0.82 * cw):] = 0
        arr[:int(0.06 * ch), :int(0.12 * cw)] = 0
        arr[:int(0.06 * ch), int(0.88 * cw):] = 0
        return Image.fromarray(arr)

class BlankCentralOvary:
    def __call__(self, img):
        arr = np.array(img, dtype=np.uint8)
        h, w = arr.shape[:2]
        arr[int(0.20*h):int(0.80*h), int(0.20*w):int(0.80*w)] = 0
        return Image.fromarray(arr)

class HarmonisedSquareCrop:
    def __call__(self, img):
        w, h = img.size
        side = int(0.70 * min(w, h))
        left, top = (w - side) // 2, (h - side) // 2
        return img.crop((left, top, left + side, top + side))

def make_transforms(mode="masked"):
    norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    if mode == "masked":
        pre = [MaskUltrasoundBorders()]
    elif mode == "ovary_blanked":
        pre = [MaskUltrasoundBorders(), BlankCentralOvary()]
    elif mode == "harmonised_crop":
        pre = [HarmonisedSquareCrop()]
    else:
        pre = []
    tr_tf = transforms.Compose(pre + [
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        norm,
    ])
    ev_tf = transforms.Compose(pre + [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        norm,
    ])
    return tr_tf, ev_tf

class PathDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths, self.labels, self.transform = list(paths), list(labels), transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return (self.transform(img) if self.transform else img), int(self.labels[i])

def set_seed(s):
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

def build_model(arch_name):
    if arch_name == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        m.classifier = nn.Linear(m.classifier.in_features, 2)
    elif arch_name == "convnext_tiny":
        m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, 2)
    elif arch_name == "mobilenet_v3_small":
        m = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, 2)
    elif arch_name == "mobilenet_v2":
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, 2)
    elif arch_name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, 2)
    elif arch_name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = nn.Linear(m.fc.in_features, 2)
    elif arch_name == "deit_tiny":
        m = timm.create_model("deit_tiny_patch16_224", pretrained=True, num_classes=2)
    else:
        raise ValueError(f"Unknown arch: {arch_name}")
    return m.to(DEVICE)

@torch.no_grad()
def evaluate_loader(model, loader):
    model.eval()
    probs, preds, labels = [], [], []
    for xb, yb in loader:
        logits = model(xb.to(DEVICE))
        p = F.softmax(logits, dim=1).cpu().numpy()
        probs.append(p)
        preds.extend(np.argmax(p, axis=1))
        labels.extend(yb.numpy())
    probs = np.vstack(probs)
    preds = np.array(preds)
    labels = np.array(labels)
    tp = int(np.sum((labels == 0) & (preds == 0)))
    fn = int(np.sum((labels == 0) & (preds == 1)))
    tn = int(np.sum((labels == 1) & (preds == 1)))
    fp = int(np.sum((labels == 1) & (preds == 0)))
    sens = 100.0 * tp / (tp + fn + 1e-12)
    spec = 100.0 * tn / (tn + fp + 1e-12)
    return {
        "acc": float(accuracy_score(labels, preds) * 100.0),
        "bal_acc": float(balanced_accuracy_score(labels, preds) * 100.0),
        "mcc": float(matthews_corrcoef(labels, preds)),
        "macro_prec": float(precision_score(labels, preds, average="macro", zero_division=0) * 100.0),
        "macro_rec": float(recall_score(labels, preds, average="macro", zero_division=0) * 100.0),
        "macro_f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
        "sens": float(sens),
        "spec": float(spec),
        "roc_auc": float(roc_auc_score((labels == 0).astype(int), probs[:, 0])),
        "tp": tp, "fn": fn, "tn": tn, "fp": fp,
    }

def train_model(
    arch, tr_ld, va_ld, epochs=20, lr=3e-4, wd=1e-4, seed=42,
    teacher=None, kd_mode="none", T=3.0, alpha=0.5, label_smoothing=0.0,
    save_path=None, record_history=False
):
    set_seed(seed)
    model = build_model(arch)
    proj = None
    params = list(model.parameters())

    if kd_mode == "feature_kd" and teacher is not None:
        t_dim = teacher.classifier.in_features if hasattr(teacher, "classifier") and isinstance(teacher.classifier, nn.Linear) else 768
        proj = nn.Linear(576, t_dim).to(DEVICE)
        params += list(proj.parameters())

    opt = torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))
    ce_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    if teacher is not None:
        teacher.eval()

    best_val_acc, best_val_loss, best_epoch, best_state = -1.0, 1e9, 0, None
    history = []

    for ep in range(1, epochs + 1):
        model.train()
        if proj is not None: proj.train()
        tr_loss_sum, n_tr = 0.0, 0
        for xb, yb in tr_ld:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                s_logits = model(xb)
                loss_ce = ce_fn(s_logits, yb)
                if teacher is not None and kd_mode != "none":
                    with torch.no_grad():
                        t_logits = teacher(xb)
                    if kd_mode == "standard":
                        loss_kd = F.kl_div(
                            F.log_softmax(s_logits / T, dim=1),
                            F.softmax(t_logits / T, dim=1),
                            reduction="batchmean"
                        ) * (T * T)
                        loss = alpha * loss_ce + (1.0 - alpha) * loss_kd
                    elif kd_mode == "logit_std":
                        s_norm = (s_logits - s_logits.mean(dim=1, keepdim=True)) / (s_logits.std(dim=1, keepdim=True) + 1e-6)
                        t_norm = (t_logits - t_logits.mean(dim=1, keepdim=True)) / (t_logits.std(dim=1, keepdim=True) + 1e-6)
                        loss_kd = F.kl_div(
                            F.log_softmax(s_norm / T, dim=1),
                            F.softmax(t_norm / T, dim=1),
                            reduction="batchmean"
                        ) * (T * T)
                        loss = alpha * loss_ce + (1.0 - alpha) * loss_kd
                    elif kd_mode == "feature_kd":
                        loss_kd = F.kl_div(
                            F.log_softmax(s_logits / T, dim=1),
                            F.softmax(t_logits / T, dim=1),
                            reduction="batchmean"
                        ) * (T * T)
                        s_feat = model.avgpool(model.features(xb)).flatten(1)
                        with torch.no_grad():
                            t_feat = F.adaptive_avg_pool2d(F.relu(teacher.features(xb)), (1, 1)).flatten(1)
                        loss_feat = F.mse_loss(F.normalize(proj(s_feat), dim=1), F.normalize(t_feat, dim=1))
                        loss = alpha * loss_ce + (1.0 - alpha) * loss_kd + 0.5 * loss_feat
                else:
                    loss = loss_ce
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tr_loss_sum += loss.item() * len(yb)
            n_tr += len(yb)
        sched.step()

        model.eval()
        va_loss_sum, va_corr, n_va = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in va_ld:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                logits = model(xb)
                l = ce_fn(logits, yb)
                va_loss_sum += l.item() * len(yb)
                va_corr += int((logits.argmax(1) == yb).sum().item())
                n_va += len(yb)
        va_acc = 100.0 * va_corr / n_va
        va_loss = va_loss_sum / n_va
        tr_loss = tr_loss_sum / n_tr

        if record_history:
            history.append({"epoch": ep, "tr_loss": tr_loss, "va_loss": va_loss, "va_acc": va_acc})

        if (va_acc > best_val_acc) or (abs(va_acc - best_val_acc) < 1e-6 and va_loss < best_val_loss):
            best_val_acc, best_val_loss, best_epoch = va_acc, va_loss, ep
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    if save_path is not None:
        torch.save(best_state, save_path)
    return model, {"best_val_acc": best_val_acc, "best_val_loss": best_val_loss, "best_epoch": best_epoch, "history": history}

tr_df = df[df["split"] == "train"].reset_index(drop=True)
va_df = df[df["split"] == "val"].reset_index(drop=True)
te_df = df[df["split"] == "test"].reset_index(drop=True)

tr_tf, ev_tf = make_transforms("masked")
tr_ld = DataLoader(PathDataset(tr_df["path"], tr_df["label"], tr_tf), batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
va_ld = DataLoader(PathDataset(va_df["path"], va_df["label"], ev_tf), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
te_ld = DataLoader(PathDataset(te_df["path"], te_df["label"], ev_tf), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

results = {
    "seeds": [42, 101, 202, 303, 404],
    "multi_seed": {},
    "seed42_histories": {},
    "baselines_seed42": {},
    "shortcut_controls_seed42": {},
    "val_sensitivity_grid": [],
    "cv10_folds": [],
}

log("\n==========================================================")
log("1. 5-SEED CORE ABLATION (CosineLR Teachers, Controls & KD)")
log("==========================================================")
CORE_MODELS = [
    "teacher_densenet121", "teacher_convnext_tiny", "standalone_mnv3",
    "label_smooth_mnv3", "self_distill_mnv3",
    "kd_densenet_mnv3", "kd_convnext_mnv3",
    "kd_logit_std_mnv3", "kd_feature_mnv3"
]
for k in CORE_MODELS:
    results["multi_seed"][k] = []

for seed in results["seeds"]:
    log(f"\n--- Running Seed {seed} ---")
    is_s42 = (seed == 42)

    t_dense, info_td = train_model(
        "densenet121", tr_ld, va_ld, epochs=20, lr=3e-4, seed=seed,
        save_path=os.path.join(CKPT_DIR, "best_teacher_densenet121.pth") if is_s42 else None,
        record_history=is_s42
    )
    m_td = evaluate_loader(t_dense, te_ld); m_td.update({"seed": seed, "best_epoch": info_td["best_epoch"], "best_val_acc": info_td["best_val_acc"]})
    results["multi_seed"]["teacher_densenet121"].append(m_td)
    if is_s42: results["seed42_histories"]["teacher_densenet121"] = info_td["history"]
    log(f"  DenseNet121 Teacher   | Ep {info_td['best_epoch']:2d} | Val {info_td['best_val_acc']:.2f}% | Test Acc {m_td['acc']:.2f}% | MCC {m_td['mcc']:.4f} | AUC {m_td['roc_auc']:.4f}")

    t_conv, info_tc = train_model(
        "convnext_tiny", tr_ld, va_ld, epochs=20, lr=2e-4, seed=seed,
        save_path=os.path.join(CKPT_DIR, "best_teacher_convnext_tiny.pth") if is_s42 else None,
        record_history=is_s42
    )
    m_tc = evaluate_loader(t_conv, te_ld); m_tc.update({"seed": seed, "best_epoch": info_tc["best_epoch"], "best_val_acc": info_tc["best_val_acc"]})
    results["multi_seed"]["teacher_convnext_tiny"].append(m_tc)
    if is_s42: results["seed42_histories"]["teacher_convnext_tiny"] = info_tc["history"]
    log(f"  ConvNeXt-Tiny Teacher | Ep {info_tc['best_epoch']:2d} | Val {info_tc['best_val_acc']:.2f}% | Test Acc {m_tc['acc']:.2f}% | MCC {m_tc['mcc']:.4f} | AUC {m_tc['roc_auc']:.4f}")

    s_std, info_ss = train_model(
        "mobilenet_v3_small", tr_ld, va_ld, epochs=20, lr=5e-4, seed=seed,
        save_path=os.path.join(CKPT_DIR, "best_standalone_mnv3.pth") if is_s42 else None,
        record_history=is_s42
    )
    m_ss = evaluate_loader(s_std, te_ld); m_ss.update({"seed": seed, "best_epoch": info_ss["best_epoch"], "best_val_acc": info_ss["best_val_acc"]})
    results["multi_seed"]["standalone_mnv3"].append(m_ss)
    if is_s42: results["seed42_histories"]["standalone_mnv3"] = info_ss["history"]
    log(f"  Standalone MNV3-Small | Ep {info_ss['best_epoch']:2d} | Val {info_ss['best_val_acc']:.2f}% | Test Acc {m_ss['acc']:.2f}% | MCC {m_ss['mcc']:.4f} | AUC {m_ss['roc_auc']:.4f}")

    s_ls, info_ls = train_model(
        "mobilenet_v3_small", tr_ld, va_ld, epochs=20, lr=5e-4, seed=seed, label_smoothing=0.1,
        save_path=os.path.join(CKPT_DIR, "best_label_smooth_mnv3.pth") if is_s42 else None
    )
    m_ls = evaluate_loader(s_ls, te_ld); m_ls.update({"seed": seed, "best_epoch": info_ls["best_epoch"], "best_val_acc": info_ls["best_val_acc"]})
    results["multi_seed"]["label_smooth_mnv3"].append(m_ls)
    log(f"  Label-Smooth (0.1)    | Ep {info_ls['best_epoch']:2d} | Val {info_ls['best_val_acc']:.2f}% | Test Acc {m_ls['acc']:.2f}% | MCC {m_ls['mcc']:.4f} | AUC {m_ls['roc_auc']:.4f}")

    s_self, info_sd = train_model(
        "mobilenet_v3_small", tr_ld, va_ld, epochs=20, lr=5e-4, seed=seed,
        teacher=s_std, kd_mode="standard", T=3.0, alpha=0.5,
        save_path=os.path.join(CKPT_DIR, "best_self_distill_mnv3.pth") if is_s42 else None
    )
    m_sd = evaluate_loader(s_self, te_ld); m_sd.update({"seed": seed, "best_epoch": info_sd["best_epoch"], "best_val_acc": info_sd["best_val_acc"]})
    results["multi_seed"]["self_distill_mnv3"].append(m_sd)
    log(f"  Self-Distill MNV3     | Ep {info_sd['best_epoch']:2d} | Val {info_sd['best_val_acc']:.2f}% | Test Acc {m_sd['acc']:.2f}% | MCC {m_sd['mcc']:.4f} | AUC {m_sd['roc_auc']:.4f}")

    s_kd1, info_kd1 = train_model(
        "mobilenet_v3_small", tr_ld, va_ld, epochs=20, lr=5e-4, seed=seed,
        teacher=t_dense, kd_mode="standard", T=3.0, alpha=0.5,
        save_path=os.path.join(CKPT_DIR, "best_kd_densenet_mnv3.pth") if is_s42 else None,
        record_history=is_s42
    )
    m_kd1 = evaluate_loader(s_kd1, te_ld); m_kd1.update({"seed": seed, "best_epoch": info_kd1["best_epoch"], "best_val_acc": info_kd1["best_val_acc"]})
    results["multi_seed"]["kd_densenet_mnv3"].append(m_kd1)
    if is_s42: results["seed42_histories"]["kd_densenet_mnv3"] = info_kd1["history"]
    log(f"  KD (DenseNet->MNV3)   | Ep {info_kd1['best_epoch']:2d} | Val {info_kd1['best_val_acc']:.2f}% | Test Acc {m_kd1['acc']:.2f}% | MCC {m_kd1['mcc']:.4f} | AUC {m_kd1['roc_auc']:.4f}")

    s_kd2, info_kd2 = train_model(
        "mobilenet_v3_small", tr_ld, va_ld, epochs=20, lr=5e-4, seed=seed,
        teacher=t_conv, kd_mode="standard", T=3.0, alpha=0.5,
        save_path=os.path.join(CKPT_DIR, "best_kd_convnext_mnv3.pth") if is_s42 else None
    )
    m_kd2 = evaluate_loader(s_kd2, te_ld); m_kd2.update({"seed": seed, "best_epoch": info_kd2["best_epoch"], "best_val_acc": info_kd2["best_val_acc"]})
    results["multi_seed"]["kd_convnext_mnv3"].append(m_kd2)
    log(f"  KD (ConvNeXt->MNV3)   | Ep {info_kd2['best_epoch']:2d} | Val {info_kd2['best_val_acc']:.2f}% | Test Acc {m_kd2['acc']:.2f}% | MCC {m_kd2['mcc']:.4f} | AUC {m_kd2['roc_auc']:.4f}")

    s_z, info_z = train_model(
        "mobilenet_v3_small", tr_ld, va_ld, epochs=20, lr=5e-4, seed=seed,
        teacher=t_dense, kd_mode="logit_std", T=3.0, alpha=0.5,
        save_path=os.path.join(CKPT_DIR, "best_kd_logit_std_mnv3.pth") if is_s42 else None
    )
    m_z = evaluate_loader(s_z, te_ld); m_z.update({"seed": seed, "best_epoch": info_z["best_epoch"], "best_val_acc": info_z["best_val_acc"]})
    results["multi_seed"]["kd_logit_std_mnv3"].append(m_z)
    log(f"  KD (Logit-Std Z-KD)   | Ep {info_z['best_epoch']:2d} | Val {info_z['best_val_acc']:.2f}% | Test Acc {m_z['acc']:.2f}% | MCC {m_z['mcc']:.4f} | AUC {m_z['roc_auc']:.4f}")

    s_f, info_f = train_model(
        "mobilenet_v3_small", tr_ld, va_ld, epochs=20, lr=5e-4, seed=seed,
        teacher=t_dense, kd_mode="feature_kd", T=3.0, alpha=0.5,
        save_path=os.path.join(CKPT_DIR, "best_kd_feature_mnv3.pth") if is_s42 else None
    )
    m_f = evaluate_loader(s_f, te_ld); m_f.update({"seed": seed, "best_epoch": info_f["best_epoch"], "best_val_acc": info_f["best_val_acc"]})
    results["multi_seed"]["kd_feature_mnv3"].append(m_f)
    log(f"  KD (Feature-Hint KD)  | Ep {info_f['best_epoch']:2d} | Val {info_f['best_val_acc']:.2f}% | Test Acc {m_f['acc']:.2f}% | MCC {m_f['mcc']:.4f} | AUC {m_f['roc_auc']:.4f}")

    if is_s42:
        t_dense_s42 = t_dense

log("\n==========================================================")
log("2. SAME-PROTOCOL BASELINES (SEED 42, 20 EPOCHS)")
log("==========================================================")
for b_arch, b_lr in [("mobilenet_v2", 5e-4), ("efficientnet_b0", 5e-4), ("resnet50", 3e-4), ("deit_tiny", 2e-4)]:
    bm, binfo = train_model(b_arch, tr_ld, va_ld, epochs=20, lr=b_lr, seed=42)
    bmet = evaluate_loader(bm, te_ld)
    bmet.update({"best_epoch": binfo["best_epoch"], "best_val_acc": binfo["best_val_acc"]})
    results["baselines_seed42"][b_arch] = bmet
    log(f"  {b_arch:<18} | Ep {binfo['best_epoch']:2d} | Val {binfo['best_val_acc']:.2f}% | Test Acc {bmet['acc']:.2f}% | BalAcc {bmet['bal_acc']:.2f}% | MCC {bmet['mcc']:.4f} | AUC {bmet['roc_auc']:.4f}")

log("\n==========================================================")
log("3. SHORTCUT CONTROLS: OVARY-BLANKED & HARMONISED CROP")
log("==========================================================")
for mode_name in ["ovary_blanked", "harmonised_crop"]:
    tr_tf_c, ev_tf_c = make_transforms(mode_name)
    tr_ld_c = DataLoader(PathDataset(tr_df["path"], tr_df["label"], tr_tf_c), batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
    va_ld_c = DataLoader(PathDataset(va_df["path"], va_df["label"], ev_tf_c), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
    te_ld_c = DataLoader(PathDataset(te_df["path"], te_df["label"], ev_tf_c), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
    cm, cinfo = train_model("mobilenet_v3_small", tr_ld_c, va_ld_c, epochs=20, lr=5e-4, seed=42)
    cmet = evaluate_loader(cm, te_ld_c)
    cmet.update({"best_epoch": cinfo["best_epoch"], "best_val_acc": cinfo["best_val_acc"]})
    results["shortcut_controls_seed42"][mode_name] = cmet
    log(f"  {mode_name:<18} | Ep {cinfo['best_epoch']:2d} | Val {cinfo['best_val_acc']:.2f}% | Test Acc {cmet['acc']:.2f}% | BalAcc {cmet['bal_acc']:.2f}% | MCC {cmet['mcc']:.4f} | AUC {cmet['roc_auc']:.4f}")

log("\n==========================================================")
log("4. VALIDATION-ONLY 5x5 (T x alpha) SENSITIVITY GRID (12 EP)")
log("==========================================================")
for T_val in [1.0, 2.0, 3.0, 5.0, 10.0]:
    for a_val in [0.1, 0.3, 0.5, 0.7, 0.9]:
        gm, ginfo = train_model(
            "mobilenet_v3_small", tr_ld, va_ld, epochs=12, lr=5e-4, seed=42,
            teacher=t_dense_s42, kd_mode="standard", T=T_val, alpha=a_val
        )
        vmet = evaluate_loader(gm, va_ld)
        results["val_sensitivity_grid"].append({"T": T_val, "alpha": a_val, "val_acc": vmet["acc"], "val_f1": vmet["macro_f1"], "val_mcc": vmet["mcc"]})
        log(f"  T={T_val:4.1f}, alpha={a_val:.1f} -> Val Acc: {vmet['acc']:.2f}% | Val F1: {vmet['macro_f1']:.4f}")

log("\n==========================================================")
log("5. 10-FOLD NESTED STRATIFIED GROUP CV (ALL 3 MODELS)")
log("==========================================================")
for fold in range(10):
    f_tr_full = df[df["cv_fold"] != fold].reset_index(drop=True)
    f_te = df[df["cv_fold"] == fold].reset_index(drop=True)
    inner_val_fold = (fold + 1) % 10
    f_tr = f_tr_full[f_tr_full["cv_fold"] != inner_val_fold].reset_index(drop=True)
    f_va = f_tr_full[f_tr_full["cv_fold"] == inner_val_fold].reset_index(drop=True)

    ld_tr = DataLoader(PathDataset(f_tr["path"], f_tr["label"], tr_tf), batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
    ld_va = DataLoader(PathDataset(f_va["path"], f_va["label"], ev_tf), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
    ld_te = DataLoader(PathDataset(f_te["path"], f_te["label"], ev_tf), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

    ft_m, _ = train_model("densenet121", ld_tr, ld_va, epochs=18, lr=3e-4, seed=42 + fold)
    fs_m, _ = train_model("mobilenet_v3_small", ld_tr, ld_va, epochs=18, lr=5e-4, seed=42 + fold)
    fk_m, _ = train_model("mobilenet_v3_small", ld_tr, ld_va, epochs=18, lr=5e-4, seed=42 + fold,
                          teacher=ft_m, kd_mode="standard", T=3.0, alpha=0.5)

    mt, ms, mk = evaluate_loader(ft_m, ld_te), evaluate_loader(fs_m, ld_te), evaluate_loader(fk_m, ld_te)
    fold_rec = {
        "fold": fold + 1,
        "n_pos": int((f_te["label"] == 0).sum()),
        "n_healthy": int((f_te["label"] == 1).sum()),
        "teacher": mt, "standalone": ms, "distilled": mk,
    }
    results["cv10_folds"].append(fold_rec)
    log(f"  Fold {fold+1:02d} (Pos={fold_rec['n_pos']}, Hlth={fold_rec['n_healthy']}) | "
        f"T={mt['acc']:.2f}% | S={ms['acc']:.2f}% | KD={mk['acc']:.2f}% (Spec={mk['spec']:.2f}%, MCC={mk['mcc']:.4f})")

t_accs = [f["teacher"]["acc"] for f in results["cv10_folds"]]
s_accs = [f["standalone"]["acc"] for f in results["cv10_folds"]]
k_accs = [f["distilled"]["acc"] for f in results["cv10_folds"]]
def safe_wilcoxon(a, b):
    diff = np.array(a) - np.array(b)
    if np.all(diff == 0): return 0.0, 1.0
    stat, p = wilcoxon(a, b, zero_method="wilcox")
    return float(stat), float(p)

w_kt, p_kt = safe_wilcoxon(k_accs, t_accs)
w_ks, p_ks = safe_wilcoxon(k_accs, s_accs)
results["cv10_wilcoxon"] = {
    "kd_vs_teacher": {"W": w_kt, "p": p_kt},
    "kd_vs_standalone": {"W": w_ks, "p": p_ks},
}
log(f"\n  10-Fold Nested CV Means: Teacher={np.mean(t_accs):.2f}±{np.std(t_accs, ddof=1):.2f}% | "
    f"Standalone={np.mean(s_accs):.2f}±{np.std(s_accs, ddof=1):.2f}% | Distilled={np.mean(k_accs):.2f}±{np.std(k_accs, ddof=1):.2f}%")
log(f"  Wilcoxon KD vs Teacher: W={w_kt:.2f}, p={p_kt:.4e} | KD vs Standalone: W={w_ks:.2f}, p={p_ks:.4e}")

elapsed_s = time.time() - t0_block2
energy_kwh = (70.0 * elapsed_s) / 3.6e6
co2e_g = energy_kwh * 475.0
results["runtime_energy"] = {"seconds": elapsed_s, "hours": elapsed_s / 3600.0, "energy_kwh": energy_kwh, "co2e_grams": co2e_g}
log(f"\nBlock 2 Runtime: {elapsed_s/3600:.2f} h ({elapsed_s:.0f} s) | Est. GPU Energy: {energy_kwh:.4f} kWh | Est. CO2e: {co2e_g:.1f} g")

with open(os.path.join(OUT_DIR, "v4_training_results.json"), "w") as f:
    json.dump(results, f, indent=2)
with open(LOG_PATH, "w") as f:
    f.write("\n".join(log_lines) + "\n")

log("BLOCK 2 COMPLETE! Proceed directly to Block 3.")


In [ ]:
# ==============================================================================
# GREENXAI-PCOS V4 — BLOCK 3: EXTERNAL VALIDATION, SPECKLE, XAI, 600-DPI FIGS & ZIP
# Paste into Kaggle Cell 3 | Runtime: ~20-30 min on T4 GPU
# Final Output: /kaggle/working/GreenXAI_V4_ALL_RESULTS.zip
# ==============================================================================

import os, json, time, copy, shutil, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from scipy.ndimage import gaussian_filter
from scipy.stats import spearmanr

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchinfo import summary as torchinfo_summary
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, confusion_matrix,
    brier_score_loss, balanced_accuracy_score, matthews_corrcoef
)
from statsmodels.stats.contingency_tables import mcnemar

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import shap
from captum.attr import IntegratedGradients

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OUT_DIR = "/kaggle/working/GreenXAI_V4_Outputs"
FIG_DIR = os.path.join(OUT_DIR, "figures_600dpi")
CKPT_DIR = os.path.join(OUT_DIR, "checkpoints")
os.makedirs(FIG_DIR, exist_ok=True)

df = pd.read_csv(os.path.join(OUT_DIR, "splits_v4_manifest.csv"))
ext_csv = os.path.join(OUT_DIR, "external_pcosgen_clean.csv")
df_ext = pd.read_csv(ext_csv) if os.path.exists(ext_csv) else pd.DataFrame(columns=["path", "label"])
with open(os.path.join(OUT_DIR, "v4_training_results.json")) as f:
    res = json.load(f)

LOG_PATH = os.path.join(OUT_DIR, "block3_eval_xai_log.txt")
log_lines = []
def log(msg=""):
    print(msg, flush=True)
    log_lines.append(str(msg))

log(f"Block 3 started | Device: {DEVICE}")

class MaskUltrasoundBorders:
    def __call__(self, img):
        w, h = img.size
        img = img.crop((int(0.03 * w), int(0.03 * h), int(0.97 * w), int(0.97 * h)))
        arr = np.array(img, dtype=np.uint8)
        ch, cw = arr.shape[:2]
        arr[int(0.90 * ch):, int(0.82 * cw):] = 0
        arr[:int(0.06 * ch), :int(0.12 * cw)] = 0
        arr[:int(0.06 * ch), int(0.88 * cw):] = 0
        return Image.fromarray(arr)

ev_tf = transforms.Compose([
    MaskUltrasoundBorders(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class PathDataset(Dataset):
    def __init__(self, paths, labels, tf): self.paths, self.labels, self.tf = list(paths), list(labels), tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return self.tf(Image.open(self.paths[i]).convert("RGB")), int(self.labels[i])

def load_arch(arch, ckpt_name):
    if arch == "densenet121":
        m = models.densenet121(weights=None); m.classifier = nn.Linear(m.classifier.in_features, 2)
    elif arch == "convnext_tiny":
        m = models.convnext_tiny(weights=None); m.classifier[2] = nn.Linear(m.classifier[2].in_features, 2)
    else:
        m = models.mobilenet_v3_small(weights=None); m.classifier[3] = nn.Linear(m.classifier[3].in_features, 2)
    m.load_state_dict(torch.load(os.path.join(CKPT_DIR, ckpt_name), map_location=DEVICE))
    return m.to(DEVICE).eval()

m_teacher_dn = load_arch("densenet121", "best_teacher_densenet121.pth")
m_teacher_cx = load_arch("convnext_tiny", "best_teacher_convnext_tiny.pth")
m_standalone = load_arch("mobilenet_v3_small", "best_standalone_mnv3.pth")
m_ls = load_arch("mobilenet_v3_small", "best_label_smooth_mnv3.pth")
m_kd_dn = load_arch("mobilenet_v3_small", "best_kd_densenet_mnv3.pth")
m_kd_cx = load_arch("mobilenet_v3_small", "best_kd_convnext_mnv3.pth")

te_df = df[df["split"] == "test"].reset_index(drop=True)
te_ld = DataLoader(PathDataset(te_df["path"], te_df["label"], ev_tf), batch_size=32, shuffle=False, num_workers=2)
te_groups = te_df["group_id"].values

@torch.no_grad()
def infer_all(model, loader):
    model.eval()
    probs, preds, labels = [], [], []
    for xb, yb in loader:
        p = F.softmax(model(xb.to(DEVICE)), dim=1).cpu().numpy()
        probs.append(p); preds.extend(np.argmax(p, axis=1)); labels.extend(yb.numpy())
    return np.vstack(probs), np.array(preds), np.array(labels)

def compute_ece(probs_pos, y_bin, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (probs_pos >= bins[i]) & (probs_pos < bins[i+1] if i < n_bins - 1 else probs_pos <= bins[i+1])
        if np.any(mask):
            ece += (np.sum(mask) / len(y_bin)) * abs(np.mean(y_bin[mask]) - np.mean(probs_pos[mask]))
    return float(ece)

def wilson_ci(k, n, z=1.95996):
    p = k / n; denom = 1 + z**2 / n
    c = (p + z**2 / (2 * n)) / denom
    h = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
    return max(0.0, (c - h) * 100.0), min(100.0, (c + h) * 100.0)

def cluster_bootstrap(y_true, y_pred, probs_pos, groups, n_boot=2000):
    rng = np.random.default_rng(SEED)
    ug = np.unique(groups)
    g2i = {g: np.where(groups == g)[0] for g in ug}
    accs, f1s, mccs, specs, aucs = [], [], [], [], []
    for _ in range(n_boot):
        sg = rng.choice(ug, size=len(ug), replace=True)
        idx = np.concatenate([g2i[g] for g in sg])
        yt, yp, pr = y_true[idx], y_pred[idx], probs_pos[idx]
        if len(np.unique(yt)) < 2: continue
        accs.append(accuracy_score(yt, yp) * 100.0)
        f1s.append(f1_score(yt, yp, average="macro"))
        mccs.append(matthews_corrcoef(yt, yp))
        tn = np.sum((yt == 1) & (yp == 1)); fp = np.sum((yt == 1) & (yp == 0))
        specs.append(100.0 * tn / (tn + fp + 1e-12))
        aucs.append(roc_auc_score((yt == 0).astype(int), pr))
    ci = lambda a: (float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5)))
    return ci(accs), ci(f1s), ci(mccs), ci(specs), ci(aucs)

log("\n==========================================================")
log("1. INTERNAL TEST SET: CLUSTER 95% CI, CALIBRATION & MCNEMAR")
log("==========================================================")
eval_models = {
    "DenseNet121 Teacher": m_teacher_dn,
    "ConvNeXt-Tiny Teacher": m_teacher_cx,
    "Standalone MNV3-Small": m_standalone,
    "Label-Smooth MNV3-Small": m_ls,
    "Distilled MNV3 (DenseNet)": m_kd_dn,
    "Distilled MNV3 (ConvNeXt)": m_kd_cx,
}
pred_cache = {}
for name, mod in eval_models.items():
    probs, preds, y_te = infer_all(mod, te_ld)
    pred_cache[name] = (probs, preds, y_te)
    y_bin = (y_te == 0).astype(int)
    p_pos = probs[:, 0]
    acc = accuracy_score(y_te, preds) * 100.0
    bacc = balanced_accuracy_score(y_te, preds) * 100.0
    mcc = matthews_corrcoef(y_te, preds)
    f1 = f1_score(y_te, preds, average="macro")
    auc = roc_auc_score(y_bin, p_pos)
    ap = average_precision_score(y_bin, p_pos)
    tp = int(np.sum((y_te == 0) & (preds == 0))); fn = int(np.sum((y_te == 0) & (preds == 1)))
    tn = int(np.sum((y_te == 1) & (preds == 1))); fp = int(np.sum((y_te == 1) & (preds == 0)))
    w_lo, w_hi = wilson_ci(tn, tn + fp)
    a_ci, f_ci, m_ci, s_ci, u_ci = cluster_bootstrap(y_te, preds, p_pos, te_groups)
    ece = compute_ece(p_pos, y_bin)
    brier = float(brier_score_loss(y_bin, p_pos))
    log(f"  {name:<26} | Acc {acc:.2f}% [{a_ci[0]:.2f},{a_ci[1]:.2f}] | BalAcc {bacc:.2f}% | MCC {mcc:.4f} [{m_ci[0]:.3f},{m_ci[1]:.3f}] | "
        f"Spec {100*tn/(tn+fp):.2f}% (Wilson [{w_lo:.2f},{w_hi:.2f}]) | AUC {auc:.4f} | AP {ap:.4f} | ECE {ece:.4f} | Brier {brier:.4f}")

y_te = pred_cache["Distilled MNV3 (DenseNet)"][2]
p_kd = pred_cache["Distilled MNV3 (DenseNet)"][1]
p_st = pred_cache["Standalone MNV3-Small"][1]
p_td = pred_cache["DenseNet121 Teacher"][1]
for lbl_pair, pa, pb in [("KD vs Standalone", p_kd, p_st), ("KD vs Teacher", p_kd, p_td)]:
    c_a = (pa == y_te); c_b = (pb == y_te)
    table = [[int(np.sum(c_a & c_b)), int(np.sum(c_a & ~c_b))], [int(np.sum(~c_a & c_b)), int(np.sum(~c_a & ~c_b))]]
    mc = mcnemar(table, exact=True)
    log(f"  McNemar {lbl_pair:<18} | b={table[0][1]}, c={table[1][0]}, p={mc.pvalue:.4f}")

log("\n==========================================================")
log("2. EXTERNAL VALIDATION (PCOSGen Deduplicated, Zero Overlap)")
log("==========================================================")
if len(df_ext) > 0:
    ext_ld = DataLoader(PathDataset(df_ext["path"], df_ext["label"], ev_tf), batch_size=32, shuffle=False, num_workers=2)
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6))
    for name, mod in eval_models.items():
        ep, epred, ey = infer_all(mod, ext_ld)
        ey_bin = (ey == 0).astype(int)
        ep_pos = ep[:, 0]
        acc = accuracy_score(ey, epred) * 100.0
        bacc = balanced_accuracy_score(ey, epred) * 100.0
        mcc = matthews_corrcoef(ey, epred)
        f1 = f1_score(ey, epred, average="macro")
        auc = roc_auc_score(ey_bin, ep_pos)
        tp = int(np.sum((ey == 0) & (epred == 0))); fn = int(np.sum((ey == 0) & (epred == 1)))
        tn = int(np.sum((ey == 1) & (epred == 1))); fp = int(np.sum((ey == 1) & (epred == 0)))
        sens, spec = 100.0*tp/(tp+fn+1e-12), 100.0*tn/(tn+fp+1e-12)
        log(f"  EXT {name:<26} | Acc {acc:.2f}% | BalAcc {bacc:.2f}% | MCC {mcc:.4f} | F1 {f1:.4f} | Sens {sens:.2f}% | Spec {spec:.2f}% | AUC {auc:.4f}")
        fpr, tpr, _ = roc_curve(ey_bin, ep_pos)
        axes[1].plot(fpr, tpr, lw=1.8, label=f"{name} (AUC={auc:.3f})")
        if name == "Distilled MNV3 (DenseNet)":
            cm_ext = confusion_matrix(ey, epred, labels=[0, 1])
            sns.heatmap(cm_ext, annot=True, fmt="d", cmap="Blues", xticklabels=["PCOM-Pos", "Healthy"], yticklabels=["PCOM-Pos", "Healthy"], ax=axes[0])
            axes[0].set_title(f"External PCOSGen CM (N={len(ey)})")
            axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
    axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5)
    axes[1].set_title("External Validation ROC Curves (PCOSGen)")
    axes[1].set_xlabel("False Positive Rate"); axes[1].set_ylabel("True Positive Rate")
    axes[1].legend(fontsize=7.5, loc="lower right"); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "external_validation_pcosgen.png"), dpi=600, bbox_inches="tight")
    plt.close()
else:
    log("  [SKIPPED] External PCOSGen CSV is empty.")

log("\n==========================================================")
log("3. SYNTHETIC RAYLEIGH SPECKLE ROBUSTNESS CURVE")
log("==========================================================")
mean_t = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
std_t = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)
sigmas = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7]
speckle_curves = {k: [] for k in ["DenseNet121 Teacher", "Standalone MNV3-Small", "Label-Smooth MNV3-Small", "Distilled MNV3 (DenseNet)"]}

for sig in sigmas:
    for mname in speckle_curves:
        mod = eval_models[mname]
        preds_s, lbls_s = [], []
        torch.manual_seed(SEED)
        with torch.no_grad():
            for xb, yb in te_ld:
                xb = xb.to(DEVICE)
                if sig > 0:
                    raw = torch.clamp(xb * std_t + mean_t, 0.0, 1.0)
                    u = torch.clamp(torch.rand_like(raw), 1e-6, 1.0 - 1e-6)
                    ray = torch.sqrt(-2.0 * torch.log(u)) / np.sqrt(np.pi / 2.0)
                    noisy = torch.clamp(raw * ((1.0 - sig) + sig * ray), 0.0, 1.0)
                    xb = (noisy - mean_t) / std_t
                preds_s.extend(mod(xb).argmax(1).cpu().numpy())
                lbls_s.extend(yb.numpy())
        speckle_curves[mname].append(accuracy_score(lbls_s, preds_s) * 100.0)
    log(f"  Speckle sigma={sig:.1f} | " + " | ".join(f"{k.split()[0]}: {speckle_curves[k][-1]:.2f}%" for k in speckle_curves))

fig, ax = plt.subplots(figsize=(6.8, 4.4))
for mname, vals in speckle_curves.items():
    ax.plot(sigmas, vals, marker="o", lw=2.0, label=mname)
ax.set_xlabel("Multiplicative Rayleigh Speckle Strength ($\\sigma$)")
ax.set_ylabel("Test Accuracy (%)")
ax.set_title("Robustness Under Synthetic Multiplicative Rayleigh Speckle")
ax.legend(fontsize=8.5); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "speckle_robustness_curve.png"), dpi=600, bbox_inches="tight")
plt.close()

log("\n==========================================================")
log("4. EXACT MACs & FP32 vs INT8 CPU LATENCY BENCHMARK")
log("==========================================================")
for name, mod in [("DenseNet121 Teacher", m_teacher_dn), ("ConvNeXt-Tiny Teacher", m_teacher_cx), ("MobileNetV3-Small Student", m_kd_dn)]:
    info = torchinfo_summary(mod, input_size=(1, 3, 224, 224), verbose=0)
    log(f"  {name:<26} | Params: {info.total_params:,} | MACs: {info.total_mult_adds/1e6:.2f} MMACs ({info.total_mult_adds/1e9:.3f} GMACs)")

dummy_cpu = torch.randn(1, 3, 224, 224)
for n_threads in [1, 4]:
    torch.set_num_threads(n_threads)
    for name, mod in [("DenseNet121 FP32", copy.deepcopy(m_teacher_dn).cpu().eval()),
                      ("MNV3-Small FP32", copy.deepcopy(m_kd_dn).cpu().eval())]:
        with torch.no_grad():
            for _ in range(15): _ = mod(dummy_cpu)
            ts = []
            for _ in range(150):
                t0 = time.perf_counter(); _ = mod(dummy_cpu); ts.append((time.perf_counter() - t0) * 1000.0)
        log(f"  CPU ({n_threads} threads) {name:<18} | {np.mean(ts):.2f} ± {np.std(ts):.2f} ms ({1000/np.mean(ts):.1f} FPS)")

torch.set_num_threads(4)
try:
    q_student = torch.ao.quantization.quantize_dynamic(copy.deepcopy(m_kd_dn).cpu().eval(), {nn.Linear}, dtype=torch.qint8)
    with torch.no_grad():
        for _ in range(15): _ = q_student(dummy_cpu)
        ts = []
        for _ in range(150):
            t0 = time.perf_counter(); _ = q_student(dummy_cpu); ts.append((time.perf_counter() - t0) * 1000.0)
    log(f"  CPU (4 threads) MNV3-Small INT8   | {np.mean(ts):.2f} ± {np.std(ts):.2f} ms ({1000/np.mean(ts):.1f} FPS)")
except Exception as e:
    log(f"  [WARN] INT8 quantisation skipped: {e}")

# Core 600-DPI Figures
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
for k, lbl, col in [("teacher_densenet121", "DenseNet121 Teacher", "#2E86C1"),
                    ("standalone_mnv3", "Standalone MNV3-Small", "#27AE60"),
                    ("kd_densenet_mnv3", "Distilled MNV3-Small", "#D35400")]:
    h = res["seed42_histories"][k]
    eps = [x["epoch"] for x in h]
    axes[0].plot(eps, [x["va_loss"] for x in h], lw=2, color=col, label=lbl)
    axes[1].plot(eps, [x["va_acc"] for x in h], lw=2, color=col, label=lbl)
axes[0].set_title("Validation Loss (Epochs 1–20)"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_title("Validation Accuracy (Epochs 1–20)"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy (%)"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "learning_curve.png"), dpi=600, bbox_inches="tight"); plt.close()

probs_kd, preds_kd, y_te = pred_cache["Distilled MNV3 (DenseNet)"]
cm = confusion_matrix(y_te, preds_kd, labels=[0, 1])
fig, ax = plt.subplots(figsize=(5.2, 4.4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["PCOM-Pos", "Healthy"], yticklabels=["PCOM-Pos", "Healthy"], ax=ax)
ax.set_title(f"Confusion Matrix (Stratified Group Test, N={len(y_te)})"); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "confusion_matrix.png"), dpi=600, bbox_inches="tight"); plt.close()

y_bin = (y_te == 0).astype(int)
fpr, tpr, _ = roc_curve(y_bin, probs_kd[:, 0])
prec_c, rec_c, _ = precision_recall_curve(y_bin, probs_kd[:, 0])
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.4))
axes[0].plot(fpr, tpr, lw=2, color="#D35400", label=f"ROC-AUC = {roc_auc_score(y_bin, probs_kd[:,0]):.4f}")
axes[0].plot([0,1],[0,1],"k--",alpha=0.4); axes[0].set_title("ROC Curve"); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(rec_c, prec_c, lw=2, color="#2980B9", label=f"PR-AUC = {average_precision_score(y_bin, probs_kd[:,0]):.4f}")
axes[1].set_title("Precision-Recall Curve"); axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "roc_pr_curves.png"), dpi=600, bbox_inches="tight"); plt.close()

gdf = pd.DataFrame(res["val_sensitivity_grid"])
piv = gdf.pivot(index="T", columns="alpha", values="val_acc")
fig, ax = plt.subplots(figsize=(6.8, 4.8))
sns.heatmap(piv, annot=True, fmt=".2f", cmap="YlGnBu", ax=ax)
ax.set_title(f"Validation-Set Sensitivity Grid (Acc %; 1 scan = {100/len(df[df['split']=='val']):.2f} pp)")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "val_sensitivity_heatmap.png"), dpi=600, bbox_inches="tight"); plt.close()

folds = [f["fold"] for f in res["cv10_folds"]]
fig, ax = plt.subplots(figsize=(8.0, 4.4))
ax.plot(folds, [f["teacher"]["acc"] for f in res["cv10_folds"]], marker="s", lw=1.8, label="DenseNet121 Teacher")
ax.plot(folds, [f["standalone"]["acc"] for f in res["cv10_folds"]], marker="^", lw=1.8, label="Standalone MNV3-Small")
ax.plot(folds, [f["distilled"]["acc"] for f in res["cv10_folds"]], marker="o", lw=2.2, label="Distilled MNV3-Small")
ax.set_xticks(folds); ax.set_xlabel("Fold"); ax.set_ylabel("Accuracy (%)")
ax.set_title("10-Fold Nested StratifiedGroupKFold Accuracy")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "10fold_cv_plot.png"), dpi=600, bbox_inches="tight"); plt.close()

feats = []
with torch.no_grad():
    for xb, _ in te_ld:
        f576 = m_kd_dn.avgpool(m_kd_dn.features(xb.to(DEVICE))).flatten(1)
        feats.append(f576.cpu().numpy())
feats = np.vstack(feats)
tsne_2d = TSNE(n_components=2, perplexity=30, learning_rate="auto", init="pca", random_state=SEED).fit_transform(feats)
top_res = te_df["res_bucket"].value_counts().head(5).index.tolist()
res_labels = [r if r in top_res else "Other" for r in te_df["res_bucket"]]

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.8))
for lbl, cname, col in [(0, "PCOM-Positive", "#E67E22"), (1, "Healthy", "#2980B9")]:
    m = (y_te == lbl)
    axes[0].scatter(tsne_2d[m, 0], tsne_2d[m, 1], s=18, alpha=0.8, color=col, label=cname)
axes[0].set_title("t-SNE by Class (perplexity=30, PCA init)"); axes[0].legend(fontsize=8.5); axes[0].grid(alpha=0.25)

for r_val in sorted(set(res_labels)):
    m = np.array([x == r_val for x in res_labels])
    axes[1].scatter(tsne_2d[m, 0], tsne_2d[m, 1], s=18, alpha=0.8, label=f"{r_val} (n={m.sum()})")
axes[1].set_title("t-SNE by Image Resolution / Acquisition Format"); axes[1].legend(fontsize=7.5); axes[1].grid(alpha=0.25)
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "tsne_class_and_source.png"), dpi=600, bbox_inches="tight"); plt.close()

err_idx = np.where(preds_kd != y_te)[0]
log(f"\nError Analysis: {len(err_idx)} test errors out of {len(y_te)} scans:")
for ei in err_idx:
    row = te_df.iloc[ei]
    log(f"  Error idx={ei} | True={y_te[ei]} Pred={preds_kd[ei]} | P(Pos)={probs_kd[ei,0]:.3f} | Res={row['res_bucket']} | File={os.path.basename(row['path'])}")
if len(err_idx) > 0:
    n_show = min(len(err_idx), 6)
    fig, axes = plt.subplots(1, n_show, figsize=(3.2 * n_show, 3.4))
    if n_show == 1: axes = [axes]
    for ax, ei in zip(axes, err_idx[:n_show]):
        row = te_df.iloc[ei]
        ax.imshow(Image.open(row["path"]).convert("RGB"))
        ax.set_title(f"T={y_te[ei]} P={preds_kd[ei]} ({probs_kd[ei,0]:.2f})\n{row['res_bucket']}", fontsize=8)
        ax.axis("off")
    plt.suptitle("Misclassified Held-Out Test Scans (Error Analysis)", fontsize=10)
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "error_analysis_grid.png"), dpi=600, bbox_inches="tight"); plt.close()

log("\n==========================================================")
log("5. XAI FAITHFULNESS (ZERO & BLUR, PER-CLASS) + ADEBAYO CHECK")
log("==========================================================")
pos_imgs, neg_imgs = [], []
for xb, yb in te_ld:
    for im, lb in zip(xb, yb.numpy()):
        if lb == 0 and len(pos_imgs) < 25: pos_imgs.append(im)
        elif lb == 1 and len(neg_imgs) < 25: neg_imgs.append(im)
    if len(pos_imgs) >= 25 and len(neg_imgs) >= 25: break

s_imgs = torch.stack(pos_imgs + neg_imgs).to(DEVICE)
s_lbls = np.array([0] * len(pos_imgs) + [1] * len(neg_imgs))

cam = GradCAM(model=m_kd_dn, target_layers=[m_kd_dn.features[-1]])
gcam_maps = np.array([cam(input_tensor=s_imgs[i:i+1], targets=[ClassifierOutputTarget(int(s_lbls[i]))])[0] for i in range(len(s_imgs))])

bg = torch.cat([s_imgs[:10], s_imgs[-10:]], dim=0)
expl = shap.GradientExplainer(m_kd_dn, bg)
shap_signed, shap_abs = [], []
for st in range(0, len(s_imgs), 10):
    sv, _ = expl.shap_values(s_imgs[st:st+10], ranked_outputs=1)
    arr = np.squeeze(np.asarray(sv[0] if isinstance(sv, list) else sv))
    if arr.ndim == 3: arr = arr[None, ...]
    shap_signed.append(np.sum(arr, axis=1))
    shap_abs.append(np.sum(np.abs(arr), axis=1))
shap_signed = np.concatenate(shap_signed, axis=0)
shap_abs = np.concatenate(shap_abs, axis=0)

ig_expl = IntegratedGradients(m_kd_dn)
ig_maps = np.array([np.sum(np.abs(ig_expl.attribute(s_imgs[i:i+1], target=int(s_lbls[i])).squeeze(0).detach().cpu().numpy()), axis=0) for i in range(len(s_imgs))])
rand_maps = np.random.rand(len(s_imgs), 224, 224)

m_rand = copy.deepcopy(m_kd_dn)
for mod in m_rand.classifier.modules():
    if isinstance(mod, nn.Linear):
        nn.init.kaiming_normal_(mod.weight)
        if mod.bias is not None: nn.init.zeros_(mod.bias)
cam_rand = GradCAM(model=m_rand, target_layers=[m_rand.features[-1]])
rhos = []
for i in range(min(20, len(s_imgs))):
    rm = cam_rand(input_tensor=s_imgs[i:i+1], targets=[ClassifierOutputTarget(int(s_lbls[i]))])[0]
    r, _ = spearmanr(gcam_maps[i].flatten(), rm.flatten())
    if not np.isnan(r): rhos.append(float(r))
log(f"  Adebayo Model-Randomisation Check (Classifier Randomised) | Mean Spearman rho = {np.mean(rhos):.4f} ± {np.std(rhos):.4f}")

def eval_ins_del(model, images, labels, maps, baseline_type="blur", steps=20):
    x_grid = np.linspace(0, 1, steps + 1)
    trapz_fn = getattr(np, "trapezoid", None) or np.trapz
    ins_aucs, del_aucs, ins_all, del_all = [], [], [], []
    for i in range(len(images)):
        img = images[i].unsqueeze(0).to(DEVICE)
        lbl = int(labels[i])
        if baseline_type == "blur":
            arr = images[i].detach().cpu().numpy()
            base = torch.tensor(np.stack([gaussian_filter(arr[c], sigma=15.0) for c in range(3)], axis=0), dtype=img.dtype, device=DEVICE).unsqueeze(0)
        else:
            base = torch.zeros_like(img)
        order = np.argsort(-maps[i].reshape(-1))
        ins_s, del_s = [], []
        with torch.no_grad():
            for s in range(steps + 1):
                k = int((s / steps) * (224 * 224))
                m = torch.zeros(224 * 224, device=DEVICE)
                if k > 0: m[order[:k]] = 1.0
                m2d = m.view(1, 1, 224, 224)
                ins_s.append(F.softmax(model(base * (1 - m2d) + img * m2d), dim=1)[0, lbl].item())
                del_s.append(F.softmax(model(img * (1 - m2d) + base * m2d), dim=1)[0, lbl].item())
        ins_all.append(ins_s); del_all.append(del_s)
        ins_aucs.append(float(trapz_fn(ins_s, x_grid))); del_aucs.append(float(trapz_fn(del_s, x_grid)))
    return x_grid, np.mean(ins_all, axis=0), np.mean(del_all, axis=0), np.array(ins_aucs), np.array(del_aucs)

fig, axes = plt.subplots(2, 2, figsize=(12.0, 8.8))
pal = {"Grad-CAM": "#D35400", "SHAP": "#2980B9", "Integrated Gradients": "#27AE60", "Random": "#7F8C8D"}
for row_i, btype in enumerate(["zero", "blur"]):
    for mname, mm in [("Grad-CAM", gcam_maps), ("SHAP", shap_abs), ("Integrated Gradients", ig_maps), ("Random", rand_maps)]:
        xg, ic, dc, ia, da = eval_ins_del(m_kd_dn, s_imgs, s_lbls, mm, baseline_type=btype)
        pos_m, neg_m = (s_lbls == 0), (s_lbls == 1)
        log(f"  XAI [{btype:4s}] {mname:<20} | Overall Ins={ia.mean():.4f}±{ia.std():.4f} Del={da.mean():.4f}±{da.std():.4f} | "
            f"Pos Ins={ia[pos_m].mean():.3f}/Del={da[pos_m].mean():.3f} | Hlth Ins={ia[neg_m].mean():.3f}/Del={da[neg_m].mean():.3f}")
        ls = "--" if mname == "Random" else "-"
        axes[row_i, 0].plot(xg*100, ic, linestyle=ls, lw=2, color=pal[mname], label=f"{mname} ({ia.mean():.3f})")
        axes[row_i, 1].plot(xg*100, dc, linestyle=ls, lw=2, color=pal[mname], label=f"{mname} ({da.mean():.3f})")
    axes[row_i, 0].set_title(f"Insertion ({btype.title()} Baseline, Higher=Better)"); axes[row_i, 0].legend(fontsize=8); axes[row_i, 0].grid(alpha=0.3)
    axes[row_i, 1].set_title(f"Deletion ({btype.title()} Baseline, Lower=Better)"); axes[row_i, 1].legend(fontsize=8); axes[row_i, 1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "xai_dual_baseline_curves.png"), dpi=600, bbox_inches="tight"); plt.close()

def denorm_img(t):
    arr = t.detach().cpu().numpy().transpose(1, 2, 0)
    return np.clip(arr * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]), 0, 1)

fig, axes = plt.subplots(3, 3, figsize=(9.5, 9.0))
for r, idx in enumerate([0, 1, 25]):
    axes[r, 0].imshow(denorm_img(s_imgs[idx])); axes[r, 0].set_title(f"{'PCOM-Pos' if s_lbls[idx]==0 else 'Healthy'} Input", fontsize=9)
    raw_s = shap_signed[idx]
    v = np.percentile(np.abs(raw_s), 99) + 1e-8
    axes[r, 1].imshow(raw_s, cmap="RdBu_r", vmin=-v, vmax=v); axes[r, 1].set_title("Unprocessed SHAP (RdBu_r)", fontsize=9)
    sm_s = gaussian_filter(np.clip(raw_s / v, -1, 1), sigma=3.0)
    axes[r, 2].imshow(sm_s, cmap="RdBu_r", vmin=-1, vmax=1); axes[r, 2].set_title("Smoothed SHAP (99th-pct)", fontsize=9)
    for c in range(3): axes[r, c].axis("off")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "shap_raw_and_smoothed.png"), dpi=600, bbox_inches="tight"); plt.close()

fig, axes = plt.subplots(2, 3, figsize=(9.5, 6.2))
for i, idx in enumerate([0, 1, 2, 25, 26, 27]):
    ax = axes[i // 3, i % 3]
    ax.imshow(denorm_img(s_imgs[idx])); ax.imshow(gcam_maps[idx], cmap="jet", alpha=0.45)
    ax.set_title(f"{'PCOM-Pos' if s_lbls[idx]==0 else 'Healthy'} Grad-CAM", fontsize=9); ax.axis("off")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "gradcam_grid.png"), dpi=600, bbox_inches="tight"); plt.close()

fig, axes = plt.subplots(2, 2, figsize=(8.2, 7.6))
for r, idx in enumerate([0, 25]):
    axes[r, 0].imshow(denorm_img(s_imgs[idx])); axes[r, 0].set_title(f"{'PCOM-Pos' if s_lbls[idx]==0 else 'Healthy'} Masked Scan", fontsize=9); axes[r, 0].axis("off")
    igm = ig_maps[idx] / (np.percentile(ig_maps[idx], 99) + 1e-8)
    axes[r, 1].imshow(denorm_img(s_imgs[idx])); axes[r, 1].imshow(np.clip(igm, 0, 1), cmap="inferno", alpha=0.55)
    axes[r, 1].set_title("Integrated Gradients Overlay", fontsize=9); axes[r, 1].axis("off")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "integrated_gradients.png"), dpi=600, bbox_inches="tight"); plt.close()

with open(LOG_PATH, "w") as f:
    f.write("\n".join(log_lines) + "\n")

master_txt = os.path.join(OUT_DIR, "v4_master_report.txt")
with open(master_txt, "w") as out_f:
    for part_name in ["block1_audit_log.txt", "block2_training_log.txt", "block3_eval_xai_log.txt"]:
        pp = os.path.join(OUT_DIR, part_name)
        if os.path.exists(pp):
            out_f.write(f"\n{'='*70}\n{part_name}\n{'='*70}\n")
            out_f.write(open(pp).read())

zip_base = "/kaggle/working/GreenXAI_V4_ALL_RESULTS"
shutil.make_archive(zip_base, "zip", OUT_DIR)
zip_size_mb = os.path.getsize(zip_base + ".zip") / (1024 * 1024)
log(f"\nALL DONE! Single downloadable archive ready: {zip_base}.zip ({zip_size_mb:.2f} MB)")


Result Inhancement , Zero-Shot

In [1]:
# ==============================================================================
# GREENXAI-PCOS V4 — BLOCK 4: ZERO-SHOT DOMAIN-GENERALISED MAIN MODEL TRAINING
# Paste into Kaggle Cell | Runtime: ~15-20 min on T4 GPU
# ==============================================================================

import os, io, glob, json, time, copy, shutil, warnings
warnings.filterwarnings("ignore")

import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, roc_curve,
    balanced_accuracy_score, matthews_corrcoef, confusion_matrix
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OUT_DIR = "/kaggle/working/GreenXAI_V4_Outputs"
FIG_DIR = os.path.join(OUT_DIR, "figures_600dpi")
CKPT_DIR = os.path.join(OUT_DIR, "checkpoints")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

def find_csv(fname):
    local = os.path.join(OUT_DIR, fname)
    if os.path.exists(local): return local
    hits = glob.glob(f"/kaggle/input/**/{fname}", recursive=True)
    if hits: return hits[0]
    raise FileNotFoundError(f"{fname} not found! Add your previous V4 notebook output as Input or run Block 1 first.")

df = pd.read_csv(find_csv("splits_v4_manifest.csv"))
df_ext = pd.read_csv(find_csv("external_pcosgen_clean.csv"))

LOG_PATH = os.path.join(OUT_DIR, "block4_domain_gen_log.txt")
log_lines = []
def log(msg=""):
    print(msg, flush=True)
    log_lines.append(str(msg))

log(f"Block 4 (Zero-Shot Domain Generalisation) started | Device: {DEVICE}")
log(f"Primary scans: {len(df)} | Clean External PCOSGen scans: {len(df_ext)}")

class UltrasoundDomainHarmonise:
    def __init__(self, crop_frac=0.65, match_res=192):
        self.crop_frac = crop_frac
        self.match_res = match_res
        self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    def __call__(self, img):
        w, h = img.size
        side = int(self.crop_frac * min(w, h))
        left, top = (w - side) // 2, (h - side) // 2
        img = img.crop((left, top, left + side, top + side))
        gray = np.array(img.convert("L"), dtype=np.uint8)
        gray = cv2.resize(gray, (self.match_res, self.match_res), interpolation=cv2.INTER_AREA)
        eq = self.clahe.apply(gray)
        rgb = np.stack([eq, eq, eq], axis=-1)
        return Image.fromarray(rgb)

class RandomUltrasoundSpeckle:
    def __init__(self, p=0.4, max_sigma=0.25):
        self.p = p
        self.max_sigma = max_sigma

    def __call__(self, tensor):
        if np.random.rand() > self.p:
            return tensor
        sig = np.random.uniform(0.05, self.max_sigma)
        u = torch.clamp(torch.rand_like(tensor), 1e-6, 1.0 - 1e-6)
        ray = torch.sqrt(-2.0 * torch.log(u)) / np.sqrt(np.pi / 2.0)
        return torch.clamp(tensor * ((1.0 - sig) + sig * ray), 0.0, 1.0)

harm_op = UltrasoundDomainHarmonise(crop_frac=0.65, match_res=192)
norm_op = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

dg_train_tf = transforms.Compose([
    harm_op,
    transforms.RandomResizedCrop(224, scale=(0.60, 1.0), ratio=(0.85, 1.15)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.2),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.35, contrast=0.35),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5))], p=0.3),
    transforms.ToTensor(),
    RandomUltrasoundSpeckle(p=0.4, max_sigma=0.25),
    norm_op,
])

dg_eval_tf = transforms.Compose([
    harm_op,
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    norm_op,
])

class PathDataset(Dataset):
    def __init__(self, paths, labels, tf):
        self.paths, self.labels, self.tf = list(paths), list(labels), tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert("RGB")), int(self.labels[i])

tr_df = df[df["split"] == "train"].reset_index(drop=True)
va_df = df[df["split"] == "val"].reset_index(drop=True)
te_df = df[df["split"] == "test"].reset_index(drop=True)

cls_counts = tr_df["label"].value_counts().to_dict()
sample_weights = [1.0 / cls_counts[int(l)] for l in tr_df["label"]]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

tr_ld = DataLoader(PathDataset(tr_df["path"], tr_df["label"], dg_train_tf), batch_size=32, sampler=sampler, num_workers=2, pin_memory=True)
va_ld = DataLoader(PathDataset(va_df["path"], va_df["label"], dg_eval_tf), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
te_ld = DataLoader(PathDataset(te_df["path"], te_df["label"], dg_eval_tf), batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
ext_ld = DataLoader(PathDataset(df_ext["path"], df_ext["label"], dg_eval_tf), batch_size=32, shuffle=False, num_workers=2)

def build_arch(name):
    if name == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        m.classifier = nn.Linear(m.classifier.in_features, 2)
    else:
        m = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, 2)
    return m.to(DEVICE)

@torch.no_grad()
def eval_model(model, loader):
    model.eval()
    probs, preds, labels = [], [], []
    for xb, yb in loader:
        p = F.softmax(model(xb.to(DEVICE)), dim=1).cpu().numpy()
        probs.append(p); preds.extend(np.argmax(p, axis=1)); labels.extend(yb.numpy())
    probs = np.vstack(probs); preds = np.array(preds); labels = np.array(labels)
    y_bin = (labels == 0).astype(int)
    tp = int(np.sum((labels == 0) & (preds == 0))); fn = int(np.sum((labels == 0) & (preds == 1)))
    tn = int(np.sum((labels == 1) & (preds == 1))); fp = int(np.sum((labels == 1) & (preds == 0)))
    return {
        "acc": float(accuracy_score(labels, preds) * 100.0),
        "bal_acc": float(balanced_accuracy_score(labels, preds) * 100.0),
        "mcc": float(matthews_corrcoef(labels, preds)),
        "f1": float(f1_score(labels, preds, average="macro", zero_division=0)),
        "sens": float(100.0 * tp / (tp + fn + 1e-12)),
        "spec": float(100.0 * tn / (tn + fp + 1e-12)),
        "auc": float(roc_auc_score(y_bin, probs[:, 0])),
        "probs": probs, "preds": preds, "labels": labels
    }

def train_dg(arch, epochs=20, lr=3e-4, teacher=None, T=3.0, alpha=0.5, save_name=None):
    torch.manual_seed(SEED); np.random.seed(SEED)
    model = build_arch(arch)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))
    ce_fn = nn.CrossEntropyLoss(label_smoothing=0.05)
    if teacher is not None: teacher.eval()

    best_score, best_ep, best_state = -1.0, 0, None
    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in tr_ld:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                sl = model(xb)
                loss_ce = ce_fn(sl, yb)
                if teacher is not None:
                    with torch.no_grad(): tl = teacher(xb)
                    loss_kd = F.kl_div(F.log_softmax(sl / T, dim=1), F.softmax(tl / T, dim=1), reduction="batchmean") * (T * T)
                    loss = alpha * loss_ce + (1.0 - alpha) * loss_kd
                else:
                    loss = loss_ce
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        sched.step()

        vm = eval_model(model, va_ld)
        if vm["bal_acc"] > best_score:
            best_score, best_ep = vm["bal_acc"], ep
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    if save_name:
        torch.save(best_state, os.path.join(CKPT_DIR, save_name))
    return model, best_ep, best_score

log("\n==========================================================")
log("1. TRAINING DOMAIN-HARMONISED MODELS (ZERO-SHOT EXTERNAL)")
log("==========================================================")
t_dg, ep_t, vb_t = train_dg("densenet121", epochs=20, lr=3e-4, save_name="best_dg_teacher_densenet121.pth")
s_dg, ep_s, vb_s = train_dg("mobilenet_v3_small", epochs=20, lr=5e-4, save_name="best_dg_standalone_mnv3.pth")
k_dg, ep_k, vb_k = train_dg("mobilenet_v3_small", epochs=20, lr=5e-4, teacher=t_dg, T=3.0, alpha=0.5, save_name="best_dg_kd_mnv3.pth")

dg_models = {
    "DG DenseNet121 Teacher": (t_dg, ep_t),
    "DG Standalone MNV3-Small": (s_dg, ep_s),
    "DG Distilled MNV3-Small": (k_dg, ep_k),
}

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6))
for name, (mod, bep) in dg_models.items():
    im = eval_model(mod, te_ld)
    em = eval_model(mod, ext_ld)
    log(f"\n--- {name} (Best Val Ep {bep}) ---")
    log(f"  INTERNAL Test (N={len(te_df)})     | Acc {im['acc']:.2f}% | BalAcc {im['bal_acc']:.2f}% | MCC {im['mcc']:.4f} | Sens {im['sens']:.2f}% | Spec {im['spec']:.2f}% | AUC {im['auc']:.4f}")
    log(f"  EXTERNAL Zero-Shot (N={len(df_ext)}) | Acc {em['acc']:.2f}% | BalAcc {em['bal_acc']:.2f}% | MCC {em['mcc']:.4f} | Sens {em['sens']:.2f}% | Spec {em['spec']:.2f}% | AUC {em['auc']:.4f}")
    fpr, tpr, _ = roc_curve((em["labels"] == 0).astype(int), em["probs"][:, 0])
    axes[1].plot(fpr, tpr, lw=2.0, label=f"{name} (AUC={em['auc']:.3f})")
    if name == "DG Distilled MNV3-Small":
        cm_e = confusion_matrix(em["labels"], em["preds"], labels=[0, 1])
        sns.heatmap(cm_e, annot=True, fmt="d", cmap="Greens", xticklabels=["PCOM-Pos", "Healthy"], yticklabels=["PCOM-Pos", "Healthy"], ax=axes[0])
        axes[0].set_title(f"Domain-Harmonised Zero-Shot CM (N={len(df_ext)})")
        axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")

axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[1].set_title("Zero-Shot External ROC (Domain-Harmonised Pipeline)")
axes[1].set_xlabel("False Positive Rate"); axes[1].set_ylabel("True Positive Rate")
axes[1].legend(fontsize=8.5, loc="lower right"); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "dg_external_zeroshot_pcosgen.png"), dpi=600, bbox_inches="tight")
plt.close()

with open(LOG_PATH, "w") as f:
    f.write("\n".join(log_lines) + "\n")

shutil.make_archive("/kaggle/working/GreenXAI_V4_Block4_DG", "zip", OUT_DIR)
log("\nBLOCK 4 COMPLETE! Saved /kaggle/working/GreenXAI_V4_Block4_DG.zip")


Block 4 (Zero-Shot Domain Generalisation) started | Device: cuda
Primary scans: 3996 | Clean External PCOSGen scans: 132

1. TRAINING DOMAIN-HARMONISED MODELS (ZERO-SHOT EXTERNAL)
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 211MB/s]


Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 189MB/s]



--- DG DenseNet121 Teacher (Best Val Ep 20) ---
  INTERNAL Test (N=328)     | Acc 96.95% | BalAcc 97.80% | MCC 0.9326 | Sens 95.59% | Spec 100.00% | AUC 0.9967
  EXTERNAL Zero-Shot (N=132) | Acc 53.03% | BalAcc 50.00% | MCC 0.0000 | Sens 100.00% | Spec 0.00% | AUC 0.2675

--- DG Standalone MNV3-Small (Best Val Ep 13) ---
  INTERNAL Test (N=328)     | Acc 95.73% | BalAcc 96.92% | MCC 0.9078 | Sens 93.83% | Spec 100.00% | AUC 0.9987
  EXTERNAL Zero-Shot (N=132) | Acc 53.79% | BalAcc 50.81% | MCC 0.0928 | Sens 100.00% | Spec 1.61% | AUC 0.3247

--- DG Distilled MNV3-Small (Best Val Ep 13) ---
  INTERNAL Test (N=328)     | Acc 94.82% | BalAcc 96.26% | MCC 0.8898 | Sens 92.51% | Spec 100.00% | AUC 0.9975
  EXTERNAL Zero-Shot (N=132) | Acc 52.27% | BalAcc 49.38% | MCC -0.0417 | Sens 97.14% | Spec 1.61% | AUC 0.3051

BLOCK 4 COMPLETE! Saved /kaggle/working/GreenXAI_V4_Block4_DG.zip
